# Aula interativa: comparar runs no MLflow e promover o Champion

Este notebook e a parte **exploratória** da aula. O pipeline reprodutível
(`src/data.py`, `src/train.py`) continua sendo a fonte da verdade; aqui
comparamos *variantes* de modelo lado a lado e escolhemos o **Champion**.

> Pré-requisito: o servidor MLflow precisa estar no ar (ver `ROTEIRO.md`,
> passo do Tracking Server).

**Contraste notebook × produção:** aqui exploramos; o que vira produção é o
que está versionado em `src/` e registrado no Registry.

In [1]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_wine
from sklearn.ensemble import GradientBoostingClassifier as GB
from sklearn.ensemble import RandomForestClassifier as RF
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split as split

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("wine-classifier")

X, y = load_wine(return_X_y=True)
Xtr, Xte, ytr, yte = split(X, y, test_size=0.2, random_state=42, stratify=y)

c:\Users\gilvan\miniconda3\envs\cev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dois runs, um único modelo registrado

Registramos `rf` e `gb` como **versões do mesmo** `wine-classifier`. Assim o
Champion vs. Challenger acontece dentro de um único modelo — sem a confusão
de ter `wine-model`, `wine-rf` etc. espalhados.

In [2]:
for nome, clf in [
    ("rf", RF(n_estimators=200, max_depth=8, random_state=42)),
    ("gb", GB(n_estimators=150, random_state=42)),
]:
    with mlflow.start_run(run_name=nome):
        mlflow.log_params(clf.get_params())
        clf.fit(Xtr, ytr)
        f1 = f1_score(yte, clf.predict(Xte), average="macro")
        mlflow.log_metric("f1_macro", f1)
        mlflow.sklearn.log_model(clf, name="model", registered_model_name="wine-classifier")
        print(f"{nome}: f1_macro={f1:.4f}")

2026/05/30 01:04:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'wine-classifier' already exists. Creating a new version of this model...
2026/05/30 01:04:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine-classifier, version 6
Created version '6' of model 'wine-classifier'.


rf: f1_macro=1.0000
🏃 View run rf at: http://127.0.0.1:5000/#/experiments/1/runs/74f8f7ee92da4f80955e84047e97e0e0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/05/30 01:04:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'wine-classifier' already exists. Creating a new version of this model...
2026/05/30 01:05:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine-classifier, version 7


gb: f1_macro=0.9453
🏃 View run gb at: http://127.0.0.1:5000/#/experiments/1/runs/728b3866b62a4ebe84af3e8b32c30c3f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '7' of model 'wine-classifier'.


## Comparar na UI

Abra <http://127.0.0.1:5000> → experimento `wine-classifier` → selecione os
runs `rf` e `gb` → botão **Compare**.

## Promover o Champion

Em vez de promover na mão, use o script versionado (escolhe a melhor por
`f1_macro`):

```powershell
uv run python src/promote.py
```

Ou force uma versão específica pela API:

In [3]:
from mlflow import MlflowClient

client = MlflowClient(tracking_uri="http://127.0.0.1:5000")
versoes = client.search_model_versions("name='wine-classifier'")
for v in sorted(versoes, key=lambda v: int(v.version)):
    f1 = client.get_run(v.run_id).data.metrics.get("f1_macro")
    print(f"v{v.version}  f1_macro={f1}  aliases={list(v.aliases)}")

# Exemplo: promover a versão 1 manualmente
# client.set_registered_model_alias('wine-classifier', 'champion', '1')

v1  f1_macro=1.0  aliases=[]
v2  f1_macro=1.0  aliases=[]
v3  f1_macro=1.0  aliases=[]
v4  f1_macro=1.0  aliases=[]
v5  f1_macro=1.0  aliases=[]
v6  f1_macro=1.0  aliases=[]
v7  f1_macro=0.9453132832080201  aliases=[]
